In [0]:
# Databricks notebook source

# ============================================================
# F1 PROJECT - INGEST SPRINTS FILES
# ============================================================
# Purpose:
# 1. Read multiple JSON files from the sprints folder
# 2. Apply an explicit schema
# 3. Combine all sprint files into one DataFrame
# 4. Add ingestion metadata
# 5. Write the data into a Bronze Delta table
# ============================================================


# COMMAND ----------

# MAGIC %run "../00_common/01_environment_config"


# COMMAND ----------

# MAGIC %run "../00_common/02_bronze_helpers"


# COMMAND ----------

# ============================================================
# Define Source Folder and Target Table
# ============================================================

source_file = f"{landing_folder_path}/sprints"

table_name = (
    f"{catalog_name}.{schema_name}.{bronze_prefix}_sprints"
)

print("Source Folder :", source_file)
print("Target Table  :", table_name)


# COMMAND ----------

# ============================================================
# Step 1 - Define Schema for Sprints JSON Files
# ============================================================

from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    FloatType
)

sprints_schema = StructType([
    StructField("date", StringType(), True),
    StructField("raceName", StringType(), True),
    StructField("round", IntegerType(), True),
    StructField("season", IntegerType(), True),
    StructField("url", StringType(), True),
    StructField("constructorId", StringType(), True),
    StructField("driverId", StringType(), True),
    StructField("grid", IntegerType(), True),
    StructField("laps", IntegerType(), True),
    StructField("number", IntegerType(), True),
    StructField("points", FloatType(), True),
    StructField("position", IntegerType(), True),
    StructField("positionText", StringType(), True),
    StructField("status", StringType(), True)
])


# COMMAND ----------

# ============================================================
# Step 2 - Read All JSON Files from Sprints Folder
# ============================================================

sprints_df = (
    spark.read
    .format("json")
    .schema(sprints_schema)
    .load(source_file)
)

display(sprints_df)


# COMMAND ----------

# ============================================================
# Step 3 - Check Schema
# ============================================================

sprints_df.printSchema()

print("Total records:", sprints_df.count())


# COMMAND ----------

# ============================================================
# Step 4 - Add Ingestion Metadata
# ============================================================

sprints_final_df = add_ingestion_metadata(sprints_df)

display(sprints_final_df)


# COMMAND ----------

# ============================================================
# Step 5 - Write to Bronze Delta Table
# ============================================================

(
    sprints_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(table_name)
)


# COMMAND ----------

# ============================================================
# Step 6 - Validate Bronze Delta Table
# ============================================================

display(
    spark.table(table_name)
)


# COMMAND ----------

print(
    "Total Bronze records:",
    spark.table(table_name).count()
)

print(
    "Bronze table created successfully:",
    table_name
)